# Auditoría y merge consolidado — v2 FINAL

Pipeline completo con SIES v2: carga → auditoría → normalización → merge → imputación → ingeniería de variables → export para modelado.

**Decisiones tomadas (validadas empíricamente):**
- Universo: `NIVEL 2 in ('Universidades CRUCH', 'Universidades Privadas', 'Universidades (* Carrera en Convenio)')`
- SIES usa código directo de su propia base (coincide 60/60 con DICCIONARIO_INSTITUCION)
- DOCENTE / INMUEBLES / ARANCELES / RETENCION usan códigos institucionales DISTINTOS de SIES, por eso se mapean vía nombre normalizado al diccionario
- 5 overrides explícitos para resolver universidades rebeldes (Bernardo O'Higgins, De O'Higgins, UNIACC, Chileno-Británica, Iberoamericana UNICIT)
- `area_conocimiento` proviene del SIES (100% cobertura)
- NIVEL 2 + NIVEL 3 ambos como features

## Sección 0 — Setup

In [1]:
## Imports y configuración de display
import pandas as pd
import numpy as np
import unicodedata
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

print('Imports OK')

Imports OK


In [22]:
## Configuración de rutas
BASE_PATH = Path(r'C:\Users\diego.perezp\Documents\GitHub\Predictive model of university admission\BBDD\BASES PROCESADAS')

RUTAS = {
    'sies':         BASE_PATH / 'SIES_FILTRADO_PARA_PILOTEAR_SOLO_PREGRADO__PROCESADO_v2.csv',
    'pib':          BASE_PATH / 'pib_regional_long.csv',
    'aranceles':    BASE_PATH / 'aranceles_puntajes_transformado.csv',
    'docente':      BASE_PATH / 'cuerpo_docente_transformado.csv',
    'inmuebles':    BASE_PATH / 'inmuebles_transformada.csv',
    'retencion':    BASE_PATH / 'retencion_empleabilidad_consolidado.csv',
    'desempleo':    BASE_PATH / 'tasa_desocupacion_transformada.csv',
    'dicc_sede':    BASE_PATH / 'diccionario_sede_region_CNED.csv',
    'dicc_inst':    BASE_PATH / 'DICCIONARIO_INSTITUCION.csv',
    'dicc_carrera': BASE_PATH / 'BBDD_NOMBRE_CARRERA.csv',
    'dicc_regiones': BASE_PATH / 'DIRCCIONARIO_REGIONES_V2.csv',
}

for nombre, ruta in RUTAS.items():
    assert ruta.exists(), f'No se encontró: {ruta}'
print(f'Las {len(RUTAS)} rutas existen y son accesibles.')

Las 11 rutas existen y son accesibles.


In [23]:
## Función de normalización básica
def normalizar(s):
    if pd.isna(s):
        return s
    s = str(s).upper().strip()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    s = s.replace('.', '')
    s = ' '.join(s.split())
    return s

assert normalizar('U. de Chile') == 'U DE CHILE'
print('normalizar() OK')

normalizar() OK


In [24]:
## Carga de archivos con fallback de encoding latin-1
dfs = {}
for nombre, ruta in RUTAS.items():
    try:
        dfs[nombre] = pd.read_csv(ruta, sep=';', decimal=',', encoding='utf-8-sig')
    except UnicodeDecodeError:
        dfs[nombre] = pd.read_csv(ruta, sep=';', decimal=',', encoding='latin-1')
        print(f'  ⚠ {nombre} cargado con encoding latin-1')
    print(f'{nombre:15s} → shape {dfs[nombre].shape}')

## Renombrar columnas del SIES v2 para consistencia con el resto del pipeline
dfs['sies'] = dfs['sies'].rename(columns={
    'CÓDIGO DE INSTITUCIÓN': 'cod_institucion',
    'ÁREA DEL CONOCIMIENTO': 'area_conocimiento',
})
print()
print(f'SIES columnas: {dfs["sies"].columns.tolist()}')

sies            → shape (630554, 12)
pib             → shape (208, 3)
aranceles       → shape (205131, 19)
docente         → shape (5723, 25)
inmuebles       → shape (5510, 10)
retencion       → shape (12711, 10)
desempleo       → shape (176, 4)
dicc_sede       → shape (380, 4)
dicc_inst       → shape (220, 2)
dicc_carrera    → shape (15940, 1)
dicc_regiones   → shape (16, 8)

SIES columnas: ['AÑO_AUX', 'REGIÓN', 'cod_institucion', 'NOMBRE INSTITUCIÓN', 'CLASIFICACIÓN INSTITUCIÓN NIVEL 2', 'CLASIFICACIÓN INSTITUCIÓN NIVEL 3', 'area_conocimiento', 'NOMBRE CARRERA', 'JORNADA', 'GENERO', 'CONDICION', 'TOTAL_MATRICULAS']


## Sección 1 — Filtrado SIES y normalización pre-merge

In [25]:
## Filtro temporal + filtro de universo
AÑO_MIN, AÑO_MAX = 2018, 2025
sies_filtrada = dfs['sies'][dfs['sies']['AÑO_AUX'].between(AÑO_MIN, AÑO_MAX)].copy()
print(f'SIES filtrado a {AÑO_MIN}-{AÑO_MAX}: {sies_filtrada.shape[0]:,} filas')

FILTRO_UNIVERSIDADES = [
    'Universidades CRUCH',
    'Universidades Privadas',
    'Universidades (* Carrera en Convenio)',
]
sies_filtrada = sies_filtrada[
    sies_filtrada['CLASIFICACIÓN INSTITUCIÓN NIVEL 2'].isin(FILTRO_UNIVERSIDADES)
].copy()
print(f'SIES filtrado a Universidades: {sies_filtrada.shape[0]:,} filas')
print(f'Universidades únicas: {sies_filtrada["cod_institucion"].nunique()}')

print('\nDistribución por NIVEL 2:')
print(sies_filtrada['CLASIFICACIÓN INSTITUCIÓN NIVEL 2'].value_counts().to_string())

## Limpiar \r\n en docente e inmuebles antes de normalizar
for nombre in ['docente', 'inmuebles']:
    col = 'Nombre Institución'
    dfs[nombre][col] = dfs[nombre][col].astype(str).str.replace(r'[\r\n]+', '', regex=True).str.strip()

## carrera_norm para joins con aranceles y retención
sies_filtrada['carrera_norm'] = sies_filtrada['NOMBRE CARRERA'].apply(normalizar)
dfs['aranceles']['carrera_norm']  = dfs['aranceles']['nombre_programa'].apply(normalizar)
dfs['retencion']['carrera_norm']  = dfs['retencion']['nombre_carrera_generica'].apply(normalizar)

print('\nNormalización de carrera_norm aplicada.')

SIES filtrado a 2018-2025: 276,878 filas
SIES filtrado a Universidades: 114,112 filas
Universidades únicas: 60

Distribución por NIVEL 2:
CLASIFICACIÓN INSTITUCIÓN NIVEL 2
Universidades Privadas                   56939
Universidades CRUCH                      56612
Universidades (* Carrera en Convenio)      561

Normalización de carrera_norm aplicada.


## Sección 2 — Normalización de regiones (cod_region)

In [27]:
## Cargar diccionario de regiones y reparar encoding doble
dicc_reg = dfs['dicc_regiones'].copy()

def reparar_encoding(s):
    if pd.isna(s):
        return s
    try:
        return s.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

col_sies_corrupta = [c for c in dicc_reg.columns if 'REGI' in c and 'Ã' in c]
if col_sies_corrupta:
    col_sies = col_sies_corrupta[0]
    dicc_reg[col_sies] = dicc_reg[col_sies].apply(reparar_encoding)
    dicc_reg['Area de referencia'] = dicc_reg['Area de referencia'].apply(reparar_encoding)
    dicc_reg = dicc_reg.rename(columns={col_sies: 'region_sies'})

dicc_reg = dicc_reg.rename(columns={
    'CodRegion':          'cod_region',
    'NombreRegion':       'region_oficial',
    'nombre_region':      'region_aranceles',
    'REGION':             'region_docente',
    'REGION.1':           'region_inmuebles',
    'region':             'region_pib',
    'Area de referencia': 'region_desempleo',
})

mask_nan = dicc_reg['region_inmuebles'].isna()
dicc_reg.loc[mask_nan, 'region_inmuebles'] = dicc_reg.loc[mask_nan, 'region_docente']

mask_nuble = dicc_reg['cod_region'] == 16
dicc_reg.loc[mask_nuble, 'region_sies']     = 'Ñuble'
dicc_reg.loc[mask_nuble, 'region_desempleo'] = 'Ñuble'

dicc_reg['cod_region'] = dicc_reg['cod_region'].astype(float).astype(int)
assert dicc_reg.shape[0] == 16

print(dicc_reg[['cod_region', 'region_oficial', 'region_sies', 'region_pib']].to_string(index=False))

 cod_region                                    region_oficial             region_sies              region_pib
         13                              Región Metropolitana           Metropolitana           Metropolitana
          8                                Región del Bío-Bío                  Biobío                  Biobío
          5                              Región de Valparaíso              Valparaíso              Valparaíso
         14                                Región de Los Ríos                Los Ríos                Los Ríos
          2                             Región de Antofagasta             Antofagasta             Antofagasta
          4                                Región de Coquimbo                Coquimbo                Coquimbo
          3                                 Región de Atacama                 Atacama                 Atacama
          9                            Región de La Araucanía            La Araucanía            La Araucanía
         1

In [8]:
## Aplicar mapping cod_region a cada fuente
COD_VIRTUAL = 99

mapping = {
    'sies':      dict(zip(dicc_reg['region_sies'],      dicc_reg['cod_region'])),
    'pib':       dict(zip(dicc_reg['region_pib'],       dicc_reg['cod_region'])),
    'aranceles': dict(zip(dicc_reg['region_aranceles'], dicc_reg['cod_region'])),
    'docente':   dict(zip(dicc_reg['region_docente'],   dicc_reg['cod_region'])),
    'inmuebles': dict(zip(dicc_reg['region_inmuebles'], dicc_reg['cod_region'])),
    'desempleo': dict(zip(dicc_reg['region_desempleo'], dicc_reg['cod_region'])),
}

mapping['docente']['Virtual'] = COD_VIRTUAL
mapping['inmuebles']['Virtual'] = COD_VIRTUAL

sies_filtrada['cod_region']    = sies_filtrada['REGIÓN'].map(mapping['sies'])
dfs['pib']['cod_region']       = dfs['pib']['region'].map(mapping['pib'])
dfs['aranceles']['cod_region'] = dfs['aranceles']['nombre_region'].map(mapping['aranceles'])
dfs['docente']['cod_region']   = dfs['docente']['REGION'].map(mapping['docente'])
dfs['inmuebles']['cod_region'] = dfs['inmuebles']['REGION'].map(mapping['inmuebles'])
dfs['desempleo']['cod_region'] = dfs['desempleo']['Area de referencia'].map(mapping['desempleo'])

print('cod_region por fuente:')
for label, df in [('SIES', sies_filtrada), ('PIB', dfs['pib']), ('ARANCELES', dfs['aranceles']),
                   ('DOCENTE', dfs['docente']), ('INMUEBLES', dfs['inmuebles']),
                   ('DESEMPLEO', dfs['desempleo'])]:
    n_null = df['cod_region'].isna().sum()
    pct = n_null / len(df) * 100 if len(df) else 0
    print(f'  {label:11s}: {n_null:>6,} nulls ({pct:5.2f}%)')

cod_region por fuente:
  SIES       :  3,147 nulls ( 2.76%)
  PIB        :      0 nulls ( 0.00%)
  ARANCELES  :      0 nulls ( 0.00%)
  DOCENTE    :      0 nulls ( 0.00%)
  INMUEBLES  :      0 nulls ( 0.00%)
  DESEMPLEO  :     11 nulls ( 6.25%)


## Sección 3 — Normalización de instituciones para fuentes CNED

**Importante:** DOCENTE, INMUEBLES, ARANCELES y RETENCION usan códigos institucionales propios (CNED), distintos del código SIES. Por eso necesitamos mapearlas vía nombre normalizado al DICCIONARIO_INSTITUCION (que sí tiene los códigos del SIES).

SIES usa su `cod_institucion` directo (validado: coincide 60/60 con el diccionario).

In [9]:
## Normalizador agresivo + 5 overrides explícitos para universidades rebeldes

ALIASES_INSTITUCION = {
    r'\bCENTRO DE FORMACION TECNICA\b': 'CFT',
    r'\bINSTITUTO PROFESIONAL\b':       'IP',
    r'\bENS\b':                          'ENSENANZA',
    r'\bEST\b':                          'ESTUDIOS',
    r'\bSUP\b':                          'SUPERIORES',
    r'\bSTGO\b':                         'SANTIAGO',
    r'\bNAC\b':                          'NACIONAL',
    r'\bESC\b':                          'ESCUELA',
}

def normalizar_institucion(s):
    if pd.isna(s):
        return s
    s = normalizar(s)
    s = re.sub(r'\bC F T\b', 'CFT', s)
    s = re.sub(r'\bI P\b',   'IP',  s)
    ## FIX v2: U sola → UNIVERSIDAD en CUALQUIER posición (no solo al inicio)
    s = re.sub(r'\bU\s', 'UNIVERSIDAD ', s)
    for pattern, replacement in ALIASES_INSTITUCION.items():
        s = re.sub(pattern, replacement, s)
    s = ' '.join(s.split())
    return s

## Cargar diccionario y limpiar convenios
dicc_inst = dfs['dicc_inst'].copy()
mask_convenio = dicc_inst['NOMBRE INSTITUCIÓN'].str.contains('CONVENIO', case=False, na=False)
dicc_inst = dicc_inst[~mask_convenio].copy()
dicc_inst = dicc_inst.rename(columns={
    'CÓDIGO DE INSTITUCIÓN': 'cod_institucion',
    'NOMBRE INSTITUCIÓN':    'nombre_institucion_canonico',
})
dicc_inst['inst_norm'] = dicc_inst['nombre_institucion_canonico'].apply(normalizar_institucion)
assert dicc_inst['inst_norm'].duplicated().sum() == 0

lookup_inst = dict(zip(dicc_inst['inst_norm'], dicc_inst['cod_institucion']))

## OVERRIDES MANUALES para universidades con naming rebelde
## (validado empíricamente: estas 5 no matchean con normalización estándar)
OVERRIDES_INSTITUCION = {
    'UNIVERSIDAD BERNARDO O`HIGGINS':                       50,
    'UNIVERSIDAD DE O`HIGGINS':                              895,
    'UNIVERSIDAD DE ARTES, CIENCIAS Y COMUNICACION UNIACC':  26,
    'UNIVERSIDAD CHILENO-BRITANICA DE CULTURA':              651,
    'UNIVERSIDAD IBEROAMERICANA DE CIENCIAS Y TECNOLOGIA':   25,
}
lookup_inst.update(OVERRIDES_INSTITUCION)

print(f'Diccionario canónico: {len(lookup_inst)} entradas ({len(OVERRIDES_INSTITUCION)} overrides incluidos)')

## Mapear cod_institucion vía nombre normalizado en las 4 fuentes CNED
dfs['aranceles']['institucion_norm'] = dfs['aranceles']['nombre_institucion'].apply(normalizar_institucion)
dfs['retencion']['institucion_norm'] = dfs['retencion']['nombre_institucion'].apply(normalizar_institucion)
dfs['docente']['institucion_norm']   = dfs['docente']['Nombre Institución'].apply(normalizar_institucion)
dfs['inmuebles']['institucion_norm'] = dfs['inmuebles']['Nombre Institución'].apply(normalizar_institucion)

dfs['aranceles']['cod_institucion'] = dfs['aranceles']['institucion_norm'].map(lookup_inst)
dfs['retencion']['cod_institucion'] = dfs['retencion']['institucion_norm'].map(lookup_inst)
dfs['docente']['cod_institucion']   = dfs['docente']['institucion_norm'].map(lookup_inst)
dfs['inmuebles']['cod_institucion'] = dfs['inmuebles']['institucion_norm'].map(lookup_inst)

## Verificación SIES-céntrica de cobertura
print('\n── Cobertura SIES-céntrica ──')
sies_codes = set(sies_filtrada['cod_institucion'].unique())
for label in ['docente', 'inmuebles', 'aranceles', 'retencion']:
    fuente_codes = set(dfs[label]['cod_institucion'].dropna().astype(int).unique())
    en_ambos = sies_codes & fuente_codes
    pct = len(en_ambos) / len(sies_codes) * 100
    print(f'  {label.upper():11s}: {len(en_ambos):>3} / {len(sies_codes)} ({pct:.1f}%)')

Diccionario canónico: 222 entradas (5 overrides incluidos)

── Cobertura SIES-céntrica ──
  DOCENTE    :  60 / 60 (100.0%)
  INMUEBLES  :  60 / 60 (100.0%)
  ARANCELES  :  60 / 60 (100.0%)
  RETENCION  :  58 / 60 (96.7%)


## Sección 4 — Auditoría de cobertura SIES-céntrica

In [10]:
## Cobertura de combinaciones de keys
def cobertura(sies_df, fuente_df, keys_sies, keys_fuente, label):
    sies_keys = sies_df[keys_sies].drop_duplicates().copy()
    sies_keys.columns = ['_k_' + str(i) for i in range(len(keys_sies))]
    fuente_keys = fuente_df[keys_fuente].drop_duplicates().copy()
    fuente_keys.columns = ['_k_' + str(i) for i in range(len(keys_fuente))]
    merged = sies_keys.merge(fuente_keys, on=list(sies_keys.columns), how='left', indicator=True)
    n_total = len(merged)
    n_match = (merged['_merge'] == 'both').sum()
    pct = n_match / n_total * 100 if n_total else 0
    return {'fuente': label, 'n_comb_sies': n_total, 'n_match': n_match,
            'n_no_match': n_total - n_match, 'pct_cobertura': round(pct, 2)}

reportes = [
    cobertura(sies_filtrada, dfs['pib'],
              ['AÑO_AUX', 'cod_region'], ['año', 'cod_region'], 'PIB'),
    cobertura(sies_filtrada, dfs['desempleo'],
              ['AÑO_AUX', 'cod_region'], ['AÑO_AUX', 'cod_region'], 'DESEMPLEO'),
    cobertura(sies_filtrada, dfs['docente'].dropna(subset=['cod_institucion']),
              ['AÑO_AUX', 'cod_region', 'cod_institucion'],
              ['Año Proceso', 'cod_region', 'cod_institucion'], 'DOCENTE'),
    cobertura(sies_filtrada, dfs['inmuebles'].dropna(subset=['cod_institucion']),
              ['AÑO_AUX', 'cod_region', 'cod_institucion'],
              ['Año Información', 'cod_region', 'cod_institucion'], 'INMUEBLES'),
    cobertura(sies_filtrada, dfs['retencion'].dropna(subset=['cod_institucion']),
              ['AÑO_AUX', 'cod_institucion', 'carrera_norm'],
              ['anio', 'cod_institucion', 'carrera_norm'], 'RETENCION'),
    cobertura(sies_filtrada, dfs['aranceles'].dropna(subset=['cod_institucion']),
              ['AÑO_AUX', 'cod_region', 'cod_institucion', 'carrera_norm'],
              ['anio', 'cod_region', 'cod_institucion', 'carrera_norm'], 'ARANCELES'),
]
df_cobertura = pd.DataFrame(reportes)
print('Cobertura SIES-céntrica:')
print(df_cobertura.to_string(index=False))

Cobertura SIES-céntrica:
   fuente  n_comb_sies  n_match  n_no_match  pct_cobertura
      PIB          128      120           8          93.75
DESEMPLEO          128      128           0         100.00
  DOCENTE         1090      842         248          77.25
INMUEBLES         1090      848         242          77.80
RETENCION        20462     4844       15618          23.67
ARANCELES        27558    17327       10231          62.87


## Sección 5 — Merge secuencial y export del consolidado

**Importante:** `area_conocimiento` viene del SIES (100% cobertura). Se elimina la columna `area_conocimiento` de aranceles antes del merge para evitar choques.

In [11]:
## Preparación pre-merge

for col in ['valor_matricula', 'valor_arancel', 'valor_titulo']:
    dfs['aranceles'][col] = pd.to_numeric(dfs['aranceles'][col], errors='coerce')

## ELIMINAR area_conocimiento de aranceles (viene del SIES)
if 'area_conocimiento' in dfs['aranceles'].columns:
    dfs['aranceles'] = dfs['aranceles'].drop(columns=['area_conocimiento'])
    print('Eliminada area_conocimiento de ARANCELES (viene del SIES)')

## Agregar aranceles a nivel de key del join
keys_aran = ['anio', 'cod_region', 'cod_institucion', 'carrera_norm']
cols_num_aran = ['duracion_semestres', 'puntaje_maximo', 'puntaje_promedio', 'puntaje_minimo',
                 'puntaje_corte_primero', 'puntaje_corte_promedio', 'puntaje_corte_ultimo',
                 'valor_matricula', 'valor_arancel', 'valor_titulo']
cols_cat_aran = ['clasificacion3', 'pregrado_posgrado']

agg_dict = {c: 'mean' for c in cols_num_aran}
agg_dict.update({c: 'first' for c in cols_cat_aran})

aran_pre = dfs['aranceles'].dropna(subset=['cod_institucion']).copy()
aran_agg = aran_pre.groupby(keys_aran, as_index=False).agg(agg_dict)
print(f'Aranceles agregados: {len(aran_pre):,} → {len(aran_agg):,} filas')

## Preparar DataFrames del lado derecho
pib_merge = dfs['pib'][['año', 'cod_region', 'pib_mm_pesos']].copy()
desemp_merge = dfs['desempleo'][['AÑO_AUX', 'cod_region', 'tasa_desempleo']].copy()

cols_docente_valores = [c for c in dfs['docente'].columns if c.startswith('N°')]
doc_merge = dfs['docente'][['Año Proceso', 'cod_region', 'cod_institucion'] + cols_docente_valores].copy()
doc_merge = doc_merge.dropna(subset=['cod_institucion'])

cols_inm_valores = [c for c in dfs['inmuebles'].columns if c.startswith('Suma')]
inm_merge = dfs['inmuebles'][['Año Información', 'cod_region', 'cod_institucion'] + cols_inm_valores].copy()
inm_merge = inm_merge.dropna(subset=['cod_institucion'])

cols_ret_valores = ['retencion_1er_anio', 'empleabilidad_1er_anio']
if 'ingreso_4to_anio_titulacion' in dfs['retencion'].columns:
    cols_ret_valores.append('ingreso_4to_anio_titulacion')
ret_merge = dfs['retencion'][['anio', 'cod_institucion', 'carrera_norm'] + cols_ret_valores].copy()
ret_merge = ret_merge.dropna(subset=['cod_institucion'])

print('Preparación pre-merge OK')

Eliminada area_conocimiento de ARANCELES (viene del SIES)
Aranceles agregados: 186,581 → 118,779 filas
Preparación pre-merge OK


In [12]:
## Merge secuencial (left join desde SIES)
consolidado = sies_filtrada.copy()
n_inicial = len(consolidado)
print(f'Inicio merge: {n_inicial:,} filas')

consolidado = consolidado.merge(
    pib_merge, left_on=['AÑO_AUX', 'cod_region'], right_on=['año', 'cod_region'], how='left'
).drop(columns=['año'])
print(f'+ PIB          → {len(consolidado):,}')

consolidado = consolidado.merge(desemp_merge, on=['AÑO_AUX', 'cod_region'], how='left')
print(f'+ DESEMPLEO    → {len(consolidado):,}')

consolidado = consolidado.merge(
    doc_merge, left_on=['AÑO_AUX', 'cod_region', 'cod_institucion'],
    right_on=['Año Proceso', 'cod_region', 'cod_institucion'], how='left'
).drop(columns=['Año Proceso'])
print(f'+ DOCENTE      → {len(consolidado):,}')

consolidado = consolidado.merge(
    inm_merge, left_on=['AÑO_AUX', 'cod_region', 'cod_institucion'],
    right_on=['Año Información', 'cod_region', 'cod_institucion'], how='left'
).drop(columns=['Año Información'])
print(f'+ INMUEBLES    → {len(consolidado):,}')

consolidado = consolidado.merge(
    ret_merge, left_on=['AÑO_AUX', 'cod_institucion', 'carrera_norm'],
    right_on=['anio', 'cod_institucion', 'carrera_norm'], how='left'
).drop(columns=['anio'])
print(f'+ RETENCION    → {len(consolidado):,}')

consolidado = consolidado.merge(
    aran_agg, left_on=['AÑO_AUX', 'cod_region', 'cod_institucion', 'carrera_norm'],
    right_on=['anio', 'cod_region', 'cod_institucion', 'carrera_norm'], how='left'
).drop(columns=['anio'])
print(f'+ ARANCELES    → {len(consolidado):,}')

assert len(consolidado) == n_inicial, f'INFLADO: {len(consolidado):,} vs {n_inicial:,}'
print(f'\n✓ Sin inflado de filas: {len(consolidado):,}')

print('\nCobertura por variable (% no-null):')
for col in consolidado.columns:
    n_null = consolidado[col].isna().sum()
    pct = (1 - n_null / len(consolidado)) * 100
    if n_null > 0:
        print(f'  {col:<40s} → {pct:6.2f}% ({n_null:>7,} nulls)')

Inicio merge: 114,112 filas
+ PIB          → 114,112
+ DESEMPLEO    → 114,112
+ DOCENTE      → 114,112
+ INMUEBLES    → 114,112
+ RETENCION    → 114,112
+ ARANCELES    → 114,112

✓ Sin inflado de filas: 114,112

Cobertura por variable (% no-null):
  cod_region                               →  97.24% (  3,147 nulls)
  pib_mm_pesos                             →  97.24% (  3,147 nulls)
  N°DocentesJornadaMedia                   →  90.04% ( 11,363 nulls)
  N°HorasMagísterJornadaHora               →  90.04% ( 11,363 nulls)
  N°HorasMagísterJornadaMedia              →  90.04% ( 11,363 nulls)
  N°HorasMagísterJornadaCompleta           →  90.04% ( 11,363 nulls)
  N°HorasDoctorJornadaHora                 →  90.04% ( 11,363 nulls)
  N°HorasDoctorJornadaMedia                →  90.04% ( 11,363 nulls)
  N°HorasDoctorJornadaCompleta             →  90.04% ( 11,363 nulls)
  N°EspecialidadesJornadaHora              →  90.04% ( 11,363 nulls)
  N°EspecialidadesJornadaMedia             →  90.04% ( 11,363 

## Sección 6 — Imputación por 5 capas jerárquicas

1. **Temporal** (docente + inmuebles): interpolación por institución+región sobre eje año
2. **Jerárquica** (aranceles + retención): cascada por institución+carrera → institución+área+año → área+año
3. **2.5 — Región+carrera** (aranceles + retención): cross-institución por carrera SIES
4. **2.7 — Institución** (aranceles + retención): institución+año → institución
5. **Terminal**: mediana global

In [13]:
## Imputación por 5 capas
df_imp = consolidado.copy()

if 'ingreso_4to_anio_titulacion' in df_imp.columns:
    df_imp = df_imp.drop(columns=['ingreso_4to_anio_titulacion'])

cols_docente   = [c for c in df_imp.columns if c.startswith('N°')]
cols_inmuebles = [c for c in df_imp.columns if c.startswith('Suma')]
cols_retencion = ['retencion_1er_anio', 'empleabilidad_1er_anio']
cols_aranceles = ['duracion_semestres', 'puntaje_maximo', 'puntaje_promedio', 'puntaje_minimo',
                  'puntaje_corte_primero', 'puntaje_corte_promedio', 'puntaje_corte_ultimo',
                  'valor_matricula', 'valor_arancel', 'valor_titulo']

cols_temporal   = cols_docente + cols_inmuebles
cols_jerarquica = cols_retencion + cols_aranceles
todas_imputar   = cols_temporal + cols_jerarquica

nulls_antes = df_imp[todas_imputar].isna()
print(f'Total celdas a imputar: {nulls_antes.sum().sum():,}')

## Capa 1: Interpolación temporal
print('\n── CAPA 1: Interpolación temporal ──')
nulls_pre = df_imp[cols_temporal].isna().sum().sum()
df_imp = df_imp.sort_values(['cod_institucion', 'cod_region', 'AÑO_AUX'])
g = df_imp.groupby(['cod_institucion', 'cod_region'])
for col in cols_temporal:
    df_imp[col] = g[col].transform(
        lambda s: s.interpolate(method='linear', limit_direction='both').ffill().bfill()
    )
print(f'  Resueltos: {nulls_pre - df_imp[cols_temporal].isna().sum().sum():,}')

## Capa 2: Jerárquica
print('\n── CAPA 2: Jerárquica ──')
def imputar_jerarquico(df, cols, niveles):
    for col in cols:
        for keys in niveles:
            if df[col].isna().sum() == 0:
                break
            medianas = df.groupby(keys)[col].transform('median')
            df[col] = df[col].fillna(medianas)
    return df

niveles_jer = [
    ['cod_institucion', 'carrera_norm'],
    ['cod_institucion', 'area_conocimiento', 'AÑO_AUX'],
    ['area_conocimiento', 'AÑO_AUX'],
]
nulls_pre = df_imp[cols_jerarquica].isna().sum().sum()
df_imp = imputar_jerarquico(df_imp, cols_jerarquica, niveles_jer)
print(f'  Resueltos: {nulls_pre - df_imp[cols_jerarquica].isna().sum().sum():,}')

## Capa 2.5: Región+carrera
print('\n── CAPA 2.5: Región+carrera ──')
niveles_carrera = [
    ['REGIÓN', 'NOMBRE CARRERA', 'AÑO_AUX'],
    ['REGIÓN', 'NOMBRE CARRERA'],
    ['NOMBRE CARRERA', 'AÑO_AUX'],
    ['NOMBRE CARRERA'],
]
nulls_pre = df_imp[cols_jerarquica].isna().sum().sum()
for keys in niveles_carrera:
    for col in cols_jerarquica:
        if df_imp[col].isna().sum() == 0:
            continue
        medianas = df_imp.groupby(keys)[col].transform('median')
        df_imp[col] = df_imp[col].fillna(medianas)
print(f'  Resueltos: {nulls_pre - df_imp[cols_jerarquica].isna().sum().sum():,}')

## Capa 2.7: Institución
print('\n── CAPA 2.7: Institución ──')
niveles_inst = [['cod_institucion', 'AÑO_AUX'], ['cod_institucion']]
nulls_pre = df_imp[cols_jerarquica].isna().sum().sum()
for keys in niveles_inst:
    for col in cols_jerarquica:
        if df_imp[col].isna().sum() == 0:
            continue
        medianas = df_imp.groupby(keys)[col].transform('median')
        df_imp[col] = df_imp[col].fillna(medianas)
print(f'  Resueltos: {nulls_pre - df_imp[cols_jerarquica].isna().sum().sum():,}')

## Capa 3: Mediana global
print('\n── CAPA 3: Mediana global ──')
nulls_pre = df_imp[todas_imputar].isna().sum().sum()
for col in todas_imputar:
    if df_imp[col].isna().sum() > 0:
        df_imp[col] = df_imp[col].fillna(df_imp[col].median())
print(f'  Resueltos: {nulls_pre - df_imp[todas_imputar].isna().sum().sum():,}')

assert df_imp[todas_imputar].isna().sum().sum() == 0

## Bandera global
df_imp['fue_imputado'] = nulls_antes.any(axis=1).reindex(df_imp.index)
print(f'\nFilas con imputación: {df_imp["fue_imputado"].sum():,} ({df_imp["fue_imputado"].mean()*100:.1f}%)')

Total celdas a imputar: 899,613

── CAPA 1: Interpolación temporal ──
  Resueltos: 37,549

── CAPA 2: Jerárquica ──
  Resueltos: 628,316

── CAPA 2.5: Región+carrera ──
  Resueltos: 0

── CAPA 2.7: Institución ──
  Resueltos: 0

── CAPA 3: Mediana global ──
  Resueltos: 233,748

Filas con imputación: 83,731 (73.4%)


## Sección 7 — Ratios docentes (composición del claustro)

In [14]:
## Construcción de 5 ratios de composición
cols_doctor      = ['N°DoctorJornadaHora', 'N°DoctorJornadaMedia', 'N°DoctorJornadaCompleta']
cols_magister    = ['N°MagisterJornadaHora', 'N°MagisterJornadaMedia', 'N°MagisterJornadaCompleta']
cols_especialista= ['N°EspecialidadesJornadaHora', 'N°EspecialidadesJornadaMedia', 'N°EspecialidadesJornadaCompleta']
cols_horas_todas = [c for c in df_imp.columns if c.startswith('N°Horas')]

suma_doctor       = df_imp[cols_doctor].sum(axis=1)
suma_magister     = df_imp[cols_magister].sum(axis=1)
suma_especialista = df_imp[cols_especialista].sum(axis=1)
suma_posgrado     = suma_doctor + suma_magister + suma_especialista

denom = df_imp['N°Docentes'].replace(0, np.nan)
df_imp['pct_doctores']      = suma_doctor / denom
df_imp['pct_magister']      = suma_magister / denom
df_imp['pct_especialistas'] = suma_especialista / denom
df_imp['pct_sin_posgrado']  = 1 - (suma_posgrado / denom)

suma_horas_completa = df_imp[['N°HorasDoctorJornadaCompleta', 'N°HorasMagísterJornadaCompleta']].sum(axis=1)
suma_horas_total = df_imp[cols_horas_todas].sum(axis=1).replace(0, np.nan)
df_imp['pct_horas_jornada_completa'] = suma_horas_completa / suma_horas_total

ratios = ['pct_doctores', 'pct_magister', 'pct_especialistas',
          'pct_sin_posgrado', 'pct_horas_jornada_completa']
for col in ratios:
    df_imp[col] = df_imp[col].fillna(0)

print('Ratios construidos:')
for col in ratios:
    print(f'  {col:<32s}: mean={df_imp[col].mean():.3f}')

Ratios construidos:
  pct_doctores                    : mean=0.201
  pct_magister                    : mean=0.390
  pct_especialistas               : mean=0.051
  pct_sin_posgrado                : mean=0.358
  pct_horas_jornada_completa      : mean=0.740


## Sección 8 — Saneamiento + export del consolidado

In [15]:
## Saneamiento del consolidado
rename_cols = {}
for col in df_imp.columns:
    col_fixed = col.replace('CLASIFICACIoN', 'CLASIFICACIÓN').replace('INSTITUCIoN', 'INSTITUCIÓN')
    if col_fixed != col:
        rename_cols[col] = col_fixed
if rename_cols:
    df_imp = df_imp.rename(columns=rename_cols)

cols_cat_nulas = ['clasificacion3', 'pregrado_posgrado',
                  'CLASIFICACIÓN INSTITUCIÓN NIVEL 2', 'CLASIFICACIÓN INSTITUCIÓN NIVEL 3']
for col in cols_cat_nulas:
    if col in df_imp.columns and df_imp[col].isna().sum() > 0:
        df_imp[col] = df_imp[col].fillna('Sin información')

n_antes = len(df_imp)
df_imp = df_imp.drop_duplicates().reset_index(drop=True)
print(f'Duplicados eliminados: {n_antes - len(df_imp):,}')

ruta_consolidado = BASE_PATH / 'sabana_consolidada_imputada_v2.csv'
df_imp.to_csv(ruta_consolidado, sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f'\n✓ Consolidado exportado: {ruta_consolidado}')
print(f'  Shape: {df_imp.shape}')

df_clean = df_imp.copy()

Duplicados eliminados: 726

✓ Consolidado exportado: C:\Users\diego.perezp\Documents\GitHub\Predictive model of university admission\BBDD\BASES PROCESADAS\sabana_consolidada_imputada_v2.csv
  Shape: (113386, 60)


## Sección 9 — Tasa de crecimiento (lag temporal)

Lag fino (institución+carrera+región+jornada+género+condición) con fallback medio. Reincorpora 2017 del SIES original para calcular el lag de 2018.

In [16]:
## Tasa de crecimiento
sies_raw = dfs['sies'].copy()
sies_raw = sies_raw[sies_raw['CLASIFICACIÓN INSTITUCIÓN NIVEL 2'].isin(FILTRO_UNIVERSIDADES)].copy()
sies_raw = sies_raw[sies_raw['AÑO_AUX'].between(2017, 2025)].copy()
print(f'SIES universidades 2017-2025: {sies_raw.shape[0]:,} filas')

KEYS_FINO = ['NOMBRE INSTITUCIÓN', 'NOMBRE CARRERA', 'REGIÓN', 'JORNADA', 'GENERO', 'CONDICION']
KEYS_MEDIO = ['NOMBRE INSTITUCIÓN', 'NOMBRE CARRERA', 'REGIÓN']

fino = sies_raw.groupby(KEYS_FINO + ['AÑO_AUX'], as_index=False)['TOTAL_MATRICULAS'].sum()
fino = fino.sort_values(KEYS_FINO + ['AÑO_AUX'])
fino['matriculas_lag1_fino'] = fino.groupby(KEYS_FINO)['TOTAL_MATRICULAS'].shift(1)
fino['anio_prev'] = fino.groupby(KEYS_FINO)['AÑO_AUX'].shift(1)
fino.loc[(fino['AÑO_AUX'] - fino['anio_prev']) != 1, 'matriculas_lag1_fino'] = np.nan

medio = (sies_raw.groupby(KEYS_MEDIO + ['AÑO_AUX'], as_index=False)['TOTAL_MATRICULAS'].sum()
                 .rename(columns={'TOTAL_MATRICULAS': 'matriculas_medio'}))
medio = medio.sort_values(KEYS_MEDIO + ['AÑO_AUX'])
medio['matriculas_lag1_medio'] = medio.groupby(KEYS_MEDIO)['matriculas_medio'].shift(1)
medio['anio_prev_m'] = medio.groupby(KEYS_MEDIO)['AÑO_AUX'].shift(1)
medio.loc[(medio['AÑO_AUX'] - medio['anio_prev_m']) != 1, 'matriculas_lag1_medio'] = np.nan

feat = fino.merge(
    medio[KEYS_MEDIO + ['AÑO_AUX', 'matriculas_lag1_medio']],
    on=KEYS_MEDIO + ['AÑO_AUX'], how='left'
)
feat['matriculas_lag1'] = feat['matriculas_lag1_fino'].fillna(feat['matriculas_lag1_medio'])
feat['crecimiento_absoluto'] = feat['TOTAL_MATRICULAS'] - feat['matriculas_lag1']
feat['tasa_crecimiento'] = np.where(
    (feat['matriculas_lag1'].notna()) & (feat['matriculas_lag1'] != 0),
    (feat['TOTAL_MATRICULAS'] - feat['matriculas_lag1']) / feat['matriculas_lag1'],
    np.nan
)
feat = feat[feat['AÑO_AUX'] >= 2018].copy()

cols_f = KEYS_FINO + ['AÑO_AUX', 'matriculas_lag1', 'crecimiento_absoluto', 'tasa_crecimiento']
n_antes_merge = len(df_clean)
df_clean = df_clean.merge(feat[cols_f], on=KEYS_FINO + ['AÑO_AUX'], how='left')
assert len(df_clean) == n_antes_merge

print(f'\n✓ Lag mergeado: {len(df_clean):,} filas')
for col in ['matriculas_lag1', 'crecimiento_absoluto', 'tasa_crecimiento']:
    pct = (1 - df_clean[col].isna().sum() / len(df_clean)) * 100
    print(f'  {col:<22s}: {pct:.1f}% cobertura')

SIES universidades 2017-2025: 129,375 filas

✓ Lag mergeado: 113,386 filas
  matriculas_lag1       : 96.3% cobertura
  crecimiento_absoluto  : 96.3% cobertura
  tasa_crecimiento      : 96.3% cobertura


## Sección 10 — Ingeniería de variables (10 features, 4 con lag para evitar leakage)

In [17]:
## 10 features de ingeniería
df_clean['_mat_inst_reg_anio'] = df_clean.groupby(
    ['NOMBRE INSTITUCIÓN', 'REGIÓN', 'AÑO_AUX'])['TOTAL_MATRICULAS'].transform('sum')
df_clean['_mat_reg_anio'] = df_clean.groupby(
    ['REGIÓN', 'AÑO_AUX'])['TOTAL_MATRICULAS'].transform('sum')

df_clean['densidad_estudiantes_m2'] = np.where(
    df_clean['Suma de M2 Construido'] > 0,
    df_clean['_mat_inst_reg_anio'] / df_clean['Suma de M2 Construido'], np.nan)

df_clean['matriculas_sobre_pib'] = np.where(
    df_clean['pib_mm_pesos'] > 0,
    df_clean['_mat_reg_anio'] / df_clean['pib_mm_pesos'], np.nan)

df_clean['ratio_docente_estudiante'] = np.where(
    df_clean['_mat_inst_reg_anio'] > 0,
    df_clean['N°Docentes'] / df_clean['_mat_inst_reg_anio'], np.nan)

p99 = df_clean['ratio_docente_estudiante'].quantile(0.99)
df_clean['ratio_docente_estudiante'] = df_clean['ratio_docente_estudiante'].clip(upper=p99)

df_clean['densidad_competitiva'] = df_clean.groupby(
    ['NOMBRE CARRERA', 'REGIÓN', 'AÑO_AUX'])['NOMBRE INSTITUCIÓN'].transform('nunique')

mediana_arancel = df_clean.groupby(['REGIÓN', 'AÑO_AUX'])['valor_arancel'].transform('median')
df_clean['arancel_relativo_regional'] = np.where(
    mediana_arancel > 0, df_clean['valor_arancel'] / mediana_arancel, np.nan)

media_puntaje = df_clean.groupby(
    ['NOMBRE INSTITUCIÓN', 'AÑO_AUX'])['puntaje_corte_ultimo'].transform('mean')
df_clean['selectividad_relativa'] = np.where(
    media_puntaje > 0, df_clean['puntaje_corte_ultimo'] / media_puntaje, np.nan)

df_clean['_mat_lag_inst'] = df_clean.groupby(
    ['NOMBRE INSTITUCIÓN', 'AÑO_AUX'])['matriculas_lag1'].transform('sum')
df_clean['_mat_lag_carrera_reg'] = df_clean.groupby(
    ['NOMBRE CARRERA', 'REGIÓN', 'AÑO_AUX'])['matriculas_lag1'].transform('sum')

df_clean['concentracion_matricula_lag'] = np.where(
    (df_clean['_mat_lag_inst'] > 0) & (df_clean['matriculas_lag1'].notna()),
    df_clean['matriculas_lag1'] / df_clean['_mat_lag_inst'], np.nan)
df_clean['pct_matricula_regional_lag'] = np.where(
    (df_clean['_mat_lag_carrera_reg'] > 0) & (df_clean['matriculas_lag1'].notna()),
    df_clean['matriculas_lag1'] / df_clean['_mat_lag_carrera_reg'], np.nan)

df_clean = df_clean.drop(columns=['_mat_inst_reg_anio', '_mat_reg_anio',
                                    '_mat_lag_inst', '_mat_lag_carrera_reg'])

print('10 features construidas')
nuevas = ['densidad_estudiantes_m2', 'matriculas_sobre_pib', 'ratio_docente_estudiante',
          'densidad_competitiva', 'arancel_relativo_regional', 'selectividad_relativa',
          'concentracion_matricula_lag', 'pct_matricula_regional_lag']
print('\nCorrelación con tasa_crecimiento:')
for col in nuevas:
    serie = df_clean[col].dropna()
    if len(serie) > 30:
        corr = serie.corr(df_clean.loc[serie.index, 'tasa_crecimiento'])
        print(f'  {col:<32s}: {corr:+.3f}')

10 features construidas

Correlación con tasa_crecimiento:
  densidad_estudiantes_m2         : +0.001
  matriculas_sobre_pib            : -0.011
  ratio_docente_estudiante        : -0.029
  densidad_competitiva            : -0.002
  arancel_relativo_regional       : +0.009
  selectividad_relativa           : -0.002
  concentracion_matricula_lag     : -0.040
  pct_matricula_regional_lag      : -0.060


## Sección 11 — Transformaciones finales y export para modelado

In [30]:
## Transformaciones finales
df_model = df_clean.copy()

cols_drop = [
    'CLASIFICACIÓN INSTITUCIÓN NIVEL 1', 'pregrado_posgrado',
    'valor_titulo', 'crecimiento_absoluto', 'TOTAL_MATRICULAS',
    'N°DocentesJornadaMedia',
    'N°HorasMagísterJornadaHora', 'N°HorasMagísterJornadaMedia', 'N°HorasMagísterJornadaCompleta',
    'N°HorasDoctorJornadaHora', 'N°HorasDoctorJornadaMedia', 'N°HorasDoctorJornadaCompleta',
    'N°EspecialidadesJornadaHora', 'N°EspecialidadesJornadaMedia', 'N°EspecialidadesJornadaCompleta',
    'N°MagisterJornadaHora', 'N°MagisterJornadaMedia', 'N°MagisterJornadaCompleta',
    'N°DoctorJornadaHora', 'N°DoctorJornadaMedia', 'N°DoctorJornadaCompleta',
    'N°DocentesHombres', 'N°DocentesMujeres',
    'Suma de M2 Terreno', 'Suma de M2 Construido', 'Suma de M2 Salas',
    'Suma de Nº Oficinas', 'Suma de Nº Salas',
    'empleabilidad_1er_anio',
    'carrera_norm', 'cod_region', 'cod_institucion', 'institucion_norm',
]
df_model = df_model.drop(columns=[c for c in cols_drop if c in df_model.columns])

## Filtro matrícula mínima
df_model = df_model[df_model['matriculas_lag1'] >= 10].copy()
print(f'Filtro lag>=10: {len(df_model):,} filas')

df_model = df_model.dropna(subset=['tasa_crecimiento']).reset_index(drop=True)
print(f'Drop sin target: {len(df_model):,} filas')

## Winsorización + arcsinh
p01 = df_model['tasa_crecimiento'].quantile(0.01)
p99 = df_model['tasa_crecimiento'].quantile(0.99)
df_model['tasa_crecimiento'] = df_model['tasa_crecimiento'].clip(lower=p01, upper=p99)
df_model['tasa_crecimiento'] = np.arcsinh(df_model['tasa_crecimiento'])

## Log-transforms
cols_log = ['matriculas_lag1', 'valor_arancel', 'valor_matricula',
            'pib_mm_pesos', 'N°Docentes', 'densidad_estudiantes_m2']
for col in cols_log:
    if col in df_model.columns:
        df_model[col] = np.log1p(df_model[col])

## Label encoding GENERO
df_model['GENERO'] = df_model['GENERO'].map({'MUJER': 0, 'HOMBRE': 1, 'NO_BINARIO': 2})

## OHE (NIVEL 2 + NIVEL 3 ambos)
cols_ohe = ['JORNADA', 'CONDICION', 'CLASIFICACIÓN INSTITUCIÓN NIVEL 2',
            'CLASIFICACIÓN INSTITUCIÓN NIVEL 3', 'clasificacion3']
cols_ohe = [c for c in cols_ohe if c in df_model.columns]
df_model = pd.get_dummies(df_model, columns=cols_ohe, drop_first=True, dtype=int)

## Target Encoding (mantener AMBAS: original + te_*)
TARGET_COL = 'tasa_crecimiento'
SMOOTHING = 30
global_mean = df_model[TARGET_COL].mean()

def target_encode(df, col, target, smoothing=30):
    stats = df.groupby(col)[target].agg(['mean', 'count'])
    smooth = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    return df[col].map(smooth)

cols_cat_originales = ['REGIÓN', 'NOMBRE INSTITUCIÓN', 'NOMBRE CARRERA', 'area_conocimiento']
for col in cols_cat_originales:
    if col in df_model.columns:
        df_model[f'te_{col}'] = target_encode(df_model, col, TARGET_COL, SMOOTHING)

df_model['fue_imputado'] = df_model['fue_imputado'].astype(int)

n_antes = len(df_model)
df_model = df_model.drop_duplicates().reset_index(drop=True)
print(f'Duplicados eliminados: {n_antes - len(df_model):,}')

## Reordenar: originales al inicio, target al final
cols_orden = (
    cols_cat_originales +
    [c for c in df_model.columns if c not in cols_cat_originales and c != TARGET_COL] +
    [TARGET_COL]
)
df_model = df_model[[c for c in cols_orden if c in df_model.columns]]

## Export
ruta_final = BASE_PATH / 'dataset_modelado_final.csv'
df_model.to_csv(ruta_final, sep=';', decimal=',', index=False, encoding='utf-8-sig')

print(f'\n── Dataset final ──')
print(f'  Shape: {df_model.shape[0]:,} filas × {df_model.shape[1]} columnas')
print(f'  Nulls: {df_model.isnull().sum().sum()}')
print(f'  Duplicados: {df_model.duplicated().sum()}')
print(f'\n✓ Exportado: {ruta_final}')

Filtro lag>=10: 88,860 filas
Drop sin target: 88,860 filas
Duplicados eliminados: 13,238

── Dataset final ──
  Shape: 75,622 filas × 51 columnas
  Nulls: 0
  Duplicados: 0

✓ Exportado: C:\Users\diego.perezp\Documents\GitHub\Predictive model of university admission\BBDD\BASES PROCESADAS\dataset_modelado_final.csv


In [29]:
## ── FIX: corregir mapping de Ñuble y re-aplicar el mapeo ───────────────────

## 1. Corregir el diccionario
mask_nuble_dicc = dicc_reg['cod_region'] == 16
dicc_reg.loc[mask_nuble_dicc, 'region_sies'] = 'Ñuble'
print('Diccionario corregido:')
print(dicc_reg[mask_nuble_dicc][['cod_region', 'region_oficial', 'region_pib', 'region_sies']].to_string(index=False))

## 2. Re-construir el mapping y re-aplicar a SIES
mapping['sies'] = dict(zip(dicc_reg['region_sies'], dicc_reg['cod_region']))
sies_filtrada['cod_region'] = sies_filtrada['REGIÓN'].map(mapping['sies'])

## 3. Verificar que Ñuble ahora tiene cod_region
n_null_antes = df_clean['cod_region'].isna().sum() if 'cod_region' in df_clean.columns else 'N/A'
print(f'\nNulls en cod_region de SIES post-fix: {sies_filtrada["cod_region"].isna().sum()}')

## 4. Re-mergear PIB en df_clean (solo para las filas de Ñuble)
##    Como ya no tenemos sies_filtrada como base de df_clean, re-mapeamos directamente
##    desde df_clean usando el nombre de la región

## Construir mapping (REGIÓN, AÑO_AUX) → pib desde la base PIB
pib_lookup = dfs['pib'].set_index(['region', 'año'])['pib_mm_pesos'].to_dict()

mask_nuble_clean = (df_clean['REGIÓN'] == 'Ñuble') & (df_clean['pib_mm_pesos'].isna())
print(f'\nFilas Ñuble sin PIB en df_clean: {mask_nuble_clean.sum():,}')

## Aplicar el lookup solo a esas filas
for idx in df_clean[mask_nuble_clean].index:
    año = df_clean.loc[idx, 'AÑO_AUX']
    pib_valor = pib_lookup.get(('Ñuble', año))
    if pib_valor is not None:
        df_clean.loc[idx, 'pib_mm_pesos'] = pib_valor

print(f'Nulls PIB post-fix: {df_clean["pib_mm_pesos"].isna().sum()}')

## 5. Recalcular matriculas_sobre_pib
df_clean['_mat_reg_anio'] = df_clean.groupby(
    ['REGIÓN', 'AÑO_AUX'])['TOTAL_MATRICULAS'].transform('sum')
df_clean['matriculas_sobre_pib'] = np.where(
    df_clean['pib_mm_pesos'] > 0,
    df_clean['_mat_reg_anio'] / df_clean['pib_mm_pesos'], np.nan
)
df_clean = df_clean.drop(columns=['_mat_reg_anio'])

print(f'Nulls matriculas_sobre_pib post-fix: {df_clean["matriculas_sobre_pib"].isna().sum()}')

Diccionario corregido:
 cod_region  region_oficial region_pib region_sies
         16 Región de Ñuble      Ñuble       Ñuble

Nulls en cod_region de SIES post-fix: 0

Filas Ñuble sin PIB en df_clean: 3,143
Nulls PIB post-fix: 0
Nulls matriculas_sobre_pib post-fix: 0


In [31]:
## ══════════════════════════════════════════════════════════════════════════
## Validación de integridad: total de matrículas por año en cada stage
## ══════════════════════════════════════════════════════════════════════════

## Stage 1: SIES crudo (todo)
stage1 = dfs['sies'].groupby('AÑO_AUX')['TOTAL_MATRICULAS'].agg(['sum', 'count']).reset_index()
stage1.columns = ['año', 'matriculas_s1_crudo', 'filas_s1_crudo']

## Stage 2: SIES filtrado a universidades y rango temporal
stage2 = sies_filtrada.groupby('AÑO_AUX')['TOTAL_MATRICULAS'].agg(['sum', 'count']).reset_index()
stage2.columns = ['año', 'matriculas_s2_filtrado', 'filas_s2_filtrado']

## Stage 3: df_clean post-merge y post-imputación (debe = stage 2)
stage3 = df_clean.groupby('AÑO_AUX')['TOTAL_MATRICULAS'].agg(['sum', 'count']).reset_index()
stage3.columns = ['año', 'matriculas_s3_postmerge', 'filas_s3_postmerge']

## Stage 4: df_clean filtrado solo a filas con tasa_crecimiento válida
stage4 = df_clean.dropna(subset=['tasa_crecimiento']).groupby('AÑO_AUX')['TOTAL_MATRICULAS'].agg(['sum', 'count']).reset_index()
stage4.columns = ['año', 'matriculas_s4_conTarget', 'filas_s4_conTarget']

## Stage 5: Dataset modelado final (df_model NO tiene TOTAL_MATRICULAS, se descartó)
## Reconstruimos las matriculas desde matriculas_lag1 + tasa_crecimiento
## matriculas_actual = matriculas_lag1 * (1 + sinh(tasa_crecimiento))
## OJO: matriculas_lag1 está en log1p, hay que revertir
matriculas_lag1_real = np.expm1(df_model['matriculas_lag1'])
tasa_real = np.sinh(df_model['tasa_crecimiento'])
df_model['_mat_recovered'] = matriculas_lag1_real * (1 + tasa_real)
stage5 = df_model.groupby('AÑO_AUX')['_mat_recovered'].agg(['sum', 'count']).reset_index()
stage5.columns = ['año', 'matriculas_s5_modelado', 'filas_s5_modelado']
df_model = df_model.drop(columns=['_mat_recovered'])

## Mergear todos los stages
comparativa = stage1.merge(stage2, on='año', how='outer') \
                    .merge(stage3, on='año', how='outer') \
                    .merge(stage4, on='año', how='outer') \
                    .merge(stage5, on='año', how='outer')

## Calcular % de retención entre stages
comparativa['pct_s2_vs_s1'] = (comparativa['matriculas_s2_filtrado'] / comparativa['matriculas_s1_crudo'] * 100).round(2)
comparativa['pct_s3_vs_s2'] = (comparativa['matriculas_s3_postmerge'] / comparativa['matriculas_s2_filtrado'] * 100).round(2)
comparativa['pct_s4_vs_s3'] = (comparativa['matriculas_s4_conTarget'] / comparativa['matriculas_s3_postmerge'] * 100).round(2)
comparativa['pct_s5_vs_s4'] = (comparativa['matriculas_s5_modelado'] / comparativa['matriculas_s4_conTarget'] * 100).round(2)

print('── Total de MATRÍCULAS por año en cada stage ──\n')
cols_mat = ['año', 'matriculas_s1_crudo', 'matriculas_s2_filtrado', 'matriculas_s3_postmerge',
            'matriculas_s4_conTarget', 'matriculas_s5_modelado']
print(comparativa[cols_mat].to_string(index=False))

print('\n── % de retención entre stages ──\n')
cols_pct = ['año', 'pct_s2_vs_s1', 'pct_s3_vs_s2', 'pct_s4_vs_s3', 'pct_s5_vs_s4']
print(comparativa[cols_pct].to_string(index=False))

print('\n── Total de FILAS por año en cada stage ──\n')
cols_filas = ['año', 'filas_s1_crudo', 'filas_s2_filtrado', 'filas_s3_postmerge',
              'filas_s4_conTarget', 'filas_s5_modelado']
print(comparativa[cols_filas].to_string(index=False))

print('\n── Totales agregados ──')
print(f'Stage 1 (SIES crudo):        {comparativa["matriculas_s1_crudo"].sum():>12,.0f} matriculas, {comparativa["filas_s1_crudo"].sum():>10,.0f} filas')
print(f'Stage 2 (SIES filtrado):     {comparativa["matriculas_s2_filtrado"].sum():>12,.0f} matriculas, {comparativa["filas_s2_filtrado"].sum():>10,.0f} filas')
print(f'Stage 3 (post-merge):        {comparativa["matriculas_s3_postmerge"].sum():>12,.0f} matriculas, {comparativa["filas_s3_postmerge"].sum():>10,.0f} filas')
print(f'Stage 4 (con target):        {comparativa["matriculas_s4_conTarget"].sum():>12,.0f} matriculas, {comparativa["filas_s4_conTarget"].sum():>10,.0f} filas')
print(f'Stage 5 (modelado final):    {comparativa["matriculas_s5_modelado"].sum():>12,.0f} matriculas, {comparativa["filas_s5_modelado"].sum():>10,.0f} filas')

── Total de MATRÍCULAS por año en cada stage ──

 año  matriculas_s1_crudo  matriculas_s2_filtrado  matriculas_s3_postmerge  matriculas_s4_conTarget  matriculas_s5_modelado
2007               748344                     NaN                      NaN                      NaN                     NaN
2008               783349                     NaN                      NaN                      NaN                     NaN
2009               849340                     NaN                      NaN                      NaN                     NaN
2010               938258                     NaN                      NaN                      NaN                     NaN
2011              1015077                     NaN                      NaN                      NaN                     NaN
2012              1064816                     NaN                      NaN                      NaN                     NaN
2013              1114277                     NaN                      NaN         